# QFT

## Introduction

The Quantum Fourier Transform algorithm is the quantum analog to the discrete fourier transform, it computes frequencies inside the input.

This is an illustration of **unitary_eq**, there is no exercise for now.

## Circuit

![Bell state circuit](../../img/qft.png)

## Verification

Verify if QFT and a modified (but equivalent) version of QFT are equivalent (with 100 qubits).

In the modified version of QFT:
- The order of RZ is reversed.
- For each wire, two X, two Z, and two Hadamard gates have been inserted at various locations in the circuit.

In [ ]:
#require "hqbricks"
open Hqbricks
open Gate_set_impl.Clifford_k
open Prog

let qft q =
  let len = qreg_len q in
  let iv = ivar "i" in
  let i_inv = len - ~$1 - iv in
  let jv = ivar "j" in
  p_for "i" ~$0 len
    ((q_idx q i_inv |> h)
    -- p_for "j" ~$1 (len - iv)
        (qbit_val q (i_inv - jv) => (q_idx q i_inv |> rz (jv + ~$1))))

let mqft q =
  let len = qreg_len q in
  let iv = ivar "i" in
  let i_inv = len - ~$1 - iv in
  let jv = ivar "j" in
  let js = ~$1 in
  let je = len - iv in
  let j_inv = je - ~$1 - jv + js in
  (q |> x) -- (q |> x)
  -- p_for "i" ~$0 len
       ((q_idx q i_inv |> h)
       -- (q_idx q i_inv |> z)
       -- (q_idx q i_inv |> z)
       -- p_for "j" js je
            (qbit_val q (i_inv - j_inv) => (q_idx q i_inv |> rz (j_inv + ~$1)))
       )
  -- (q |> h) -- (q |> h)

let len = 100
let q = qreg "q" ~$len
let qft_circ = qft q
let mqft_circ = mqft q
let () =
  print_endline "\nQFT circ:";
  Prog.print qft_circ;
  print_endline "\nMQFT circ:";
  Prog.print mqft_circ;
  print_endline ""

let input_hps = Hps.(Stdlib.(one |> add_qmem_vec_x ("q", 0) len 0))
let () =
  print_endline "\nInput HPS:";
  print_endline (Hps.to_string input_hps);
  print_endline ""

let () =
  assert (Assertion.unitary_eq qft_circ mqft_circ input_hps);
  print_endline "Equivalence Verified"